# Diabetes Project - Feature Engineering + Model Building
**Dataset:** cleaned_diabetes_data.csv | **Rows:** 101,763 | **Target:** readmitted (<30 days = 1)

This notebook covers:
1. Feature Engineering
2. Model Building (Logistic Regression + Random Forest)
3. Model Evaluation (ROC-AUC, Confusion Matrix, Feature Importance)

---

## Step 1 - Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, roc_curve)
from sklearn.preprocessing import LabelEncoder

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('cleaned_diabetes_data.csv')
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(3)

## Step 2 - Quick Clean

In [ ]:
df['max_glu_serum'] = df['max_glu_serum'].fillna('None')
df['A1Cresult']     = df['A1Cresult'].fillna('None')
for col in ['race', 'diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].replace('?', 'Unknown')
print('Cleaned. Nulls remaining:', df.isnull().sum().sum())

## Step 3 - Create Binary Target
> **Goal:** Predict if a patient is re-admitted within 30 days (`<30` = 1, everything else = 0)

> **Why binary?** Models need numbers. Early readmission (<30 days) is the most critical clinical outcome to predict.

In [ ]:
df['readmit_binary'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

counts = df['readmit_binary'].value_counts()
print('Target distribution:')
print(f'  0 (Not early readmitted)  : {counts[0]:,}  ({counts[0]/len(df)*100:.1f}%)')
print(f'  1 (Readmitted in <30 days): {counts[1]:,}  ({counts[1]/len(df)*100:.1f}%)')
print()
print('Class Imbalance: 11.2% positive class.')
print('We will use class_weight=balanced in models.')

## Step 4 - Simplify 23 Drug Columns
Each drug column (metformin, insulin, glipizide etc.) has values: `No / Steady / Up / Down`.
We convert to **binary** -- was the drug active or not?

In [ ]:
drug_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

# 0 = not active (No), 1 = active (Steady/Up/Down)
for col in drug_cols:
    df[col + '_active'] = df[col].apply(lambda x: 0 if x == 'No' else 1)

# Special flag: was insulin dose CHANGED?
df['insulin_changed'] = df['insulin'].apply(lambda x: 1 if x in ['Up', 'Down'] else 0)

# Total active drugs per patient (new useful feature!)
active_cols = [c + '_active' for c in drug_cols]
df['total_active_drugs'] = df[active_cols].sum(axis=1)

print(f'Created {len(active_cols)} drug active flags')
print(f'Created insulin_changed flag')
print(f'Created total_active_drugs feature')
print(f'\nTotal active drugs stats:')
print(df['total_active_drugs'].describe().round(1))

## Step 5 - Create Age Group Feature
Age is stored as decade midpoints (5=0-10yrs, 65=60-70yrs). We create readable groups.

In [ ]:
df['age_group'] = pd.cut(df['age'],
    bins=[0, 30, 50, 70, 100],
    labels=['Young(0-30)', 'Middle(30-50)', 'Senior(50-70)', 'Elderly(70+)']
)

print('Age group distribution:')
print(df['age_group'].value_counts().sort_index())

plt.figure(figsize=(8, 4))
df.groupby('age_group', observed=True)['readmit_binary'].mean().mul(100).plot(
    kind='bar', color=['#3498DB','#2ECC71','#E67E22','#E74C3C'], edgecolor='none')
plt.title('Early Readmission Rate (%) by Age Group', fontweight='bold')
plt.ylabel('Readmission Rate (%)')
plt.xlabel('Age Group')
plt.xticks(rotation=0)
for i, v in enumerate(df.groupby('age_group', observed=True)['readmit_binary'].mean().mul(100)):
    plt.text(i, v + 0.1, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('FE_01_age_readmission_rate.png', dpi=150, bbox_inches='tight')
plt.show()
print('Elderly patients (70+) have the highest early readmission rate!')

## Step 6 - Encode Categorical Columns
ML models only understand numbers. We convert text columns to numbers.

In [ ]:
le = LabelEncoder()

df['gender_enc']      = le.fit_transform(df['gender'])
df['change_enc']      = df['change'].map({'No': 0, 'Ch': 1})
df['diabetesMed_enc'] = df['diabetesMed'].map({'No': 0, 'Yes': 1})
df['race_enc']        = le.fit_transform(df['race'])
df['A1Cresult_enc']   = le.fit_transform(df['A1Cresult'])
df['max_glu_enc']     = le.fit_transform(df['max_glu_serum'])

age_map = {'Young(0-30)': 0, 'Middle(30-50)': 1, 'Senior(50-70)': 2, 'Elderly(70+)': 3}
df['age_group_enc'] = df['age_group'].map(age_map)

print('Encoded columns created successfully!')
encoded = ['gender_enc','change_enc','diabetesMed_enc','race_enc','A1Cresult_enc','max_glu_enc','age_group_enc']
for c in encoded:
    print(f'  {c}: {sorted(df[c].unique())}')

## Step 7 - Select Final Features
We pick the most meaningful features and drop original text columns.

In [ ]:
numeric_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses', 'total_active_drugs',
    'insulin_changed', 'age'
]

encoded_features = [
    'gender_enc', 'race_enc', 'change_enc', 'diabetesMed_enc',
    'A1Cresult_enc', 'max_glu_enc', 'age_group_enc',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id'
]

drug_active_features = [col + '_active' for col in drug_cols]

all_features = numeric_features + encoded_features + drug_active_features

X = df[all_features]
y = df['readmit_binary']

print(f'Final feature set: {X.shape[1]} features')
print(f'Target samples   : {y.shape[0]:,}')
print(f'Null check       : {X.isnull().sum().sum()} nulls in features')

## Step 8 - Train/Test Split
**80% training, 20% testing.** `stratify=y` ensures same class ratio in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train/Test Split:')
print(f'  X_train : {X_train.shape[0]:,} rows x {X_train.shape[1]} features')
print(f'  X_test  : {X_test.shape[0]:,} rows x {X_test.shape[1]} features')
print(f'\nClass balance in train set:')
print(f'  0 (no early readmit): {(y_train==0).sum():,}  ({(y_train==0).mean()*100:.1f}%)')
print(f'  1 (early readmit)   : {(y_train==1).sum():,}  ({(y_train==1).mean()*100:.1f}%)')

## Step 9 - Model 1: Logistic Regression
Simple, interpretable baseline. `class_weight='balanced'` handles class imbalance.

In [ ]:
print('Training Logistic Regression...')
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train, y_train)

lr_pred  = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]
lr_auc   = roc_auc_score(y_test, lr_proba)

print('\nLogistic Regression Results:')
print('=' * 45)
print(classification_report(y_test, lr_pred, target_names=['No Early Readmit', 'Early Readmit']))
print(f'ROC-AUC Score : {lr_auc:.4f}')
print()
print('How to read:')
print('  Precision = Of patients flagged positive, how many truly were?')
print('  Recall    = Of all actual positives, how many did we catch?')
print('  ROC-AUC   = 0.5 means random | 1.0 means perfect model')

## Step 10 - Model 2: Random Forest
More powerful -- builds 100 decision trees and combines results. Usually beats Logistic Regression.

In [ ]:
print('Training Random Forest (may take 1-2 minutes)...')
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_pred  = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_auc   = roc_auc_score(y_test, rf_proba)

print('\nRandom Forest Results:')
print('=' * 45)
print(classification_report(y_test, rf_pred, target_names=['No Early Readmit', 'Early Readmit']))
print(f'ROC-AUC Score : {rf_auc:.4f}')

## Step 11 - Compare Both Models

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

lr_acc = accuracy_score(y_test, lr_pred)
rf_acc = accuracy_score(y_test, rf_pred)
lr_f1  = f1_score(y_test, lr_pred)
rf_f1  = f1_score(y_test, rf_pred)

print('=' * 55)
print('MODEL COMPARISON')
print('=' * 55)
print(f'{"Metric":<20} {"Logistic Reg":>16} {"Random Forest":>15}')
print('-' * 55)
print(f'{"ROC-AUC":<20} {lr_auc:>16.4f} {rf_auc:>15.4f}')
print(f'{"Accuracy":<20} {lr_acc:>16.4f} {rf_acc:>15.4f}')
print(f'{"F1 Score (pos)":<20} {lr_f1:>16.4f} {rf_f1:>15.4f}')
print()
winner = 'Random Forest' if rf_auc > lr_auc else 'Logistic Regression'
print(f'Best Model: {winner} (higher ROC-AUC)')

## Step 12 - Confusion Matrix
Shows correct vs wrong predictions for each class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')

for ax, pred, title in [
    (axes[0], lr_pred, 'Logistic Regression'),
    (axes[1], rf_pred, 'Random Forest')
]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Predicted No', 'Predicted Yes'],
                yticklabels=['Actual No', 'Actual Yes'],
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('MODEL_01_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('Reading the matrix:')
print('  Top-left     = True Negatives  (correctly said not readmitted)')
print('  Bottom-right = True Positives  (correctly caught early readmissions)')
print('  Top-right    = False Positives (wrongly flagged as readmitted)')
print('  Bottom-left  = False Negatives (missed readmissions -- most costly!)')

## Step 13 - ROC Curve
Shows how well your model separates the two classes. AUC closer to 1.0 = better model.

In [ ]:
plt.figure(figsize=(8, 6))

for proba, label, color in [
    (lr_proba, f'Logistic Regression (AUC={lr_auc:.3f})', '#3498DB'),
    (rf_proba, f'Random Forest       (AUC={rf_auc:.3f})', '#E74C3C')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, color=color, linewidth=2.5, label=label)

plt.plot([0,1], [0,1], 'k--', linewidth=1, label='Random Guess (AUC=0.5)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('ROC Curve -- Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('MODEL_02_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 14 - Feature Importance
Which features had the most influence on predicting early readmission?

In [ ]:
feat_imp = pd.Series(
    rf_model.feature_importances_, index=X.columns
).sort_values(ascending=False)

top15 = feat_imp.head(15)

plt.figure(figsize=(10, 7))
colors = ['#E74C3C' if i < 3 else '#3498DB' if i < 8 else '#95A5A6' for i in range(len(top15))]
plt.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1], edgecolor='none')
plt.xlabel('Feature Importance Score', fontsize=12)
plt.title('Top 15 Most Important Features (Random Forest)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('MODEL_03_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 most important features:')
for i, (feat, score) in enumerate(feat_imp.head(10).items(), 1):
    print(f'  {i:>2}. {feat:<35} {score:.4f}')

## Step 15 - Save Your Best Model

In [ ]:
import joblib

# Save the Random Forest model
joblib.dump(rf_model, 'diabetes_rf_model.pkl')
print('Model saved as diabetes_rf_model.pkl')

# How to load it later:
# loaded_model = joblib.load('diabetes_rf_model.pkl')
# predictions  = loaded_model.predict(X_test)

## Step 16 - Final Summary


In [ ]:
print('=' * 60)
print('PROJECT SUMMARY - DIABETES READMISSION PREDICTION')
print('=' * 60)
print()
print('DATASET')
print(f'  Rows     : 101,763 patients')
print(f'  Features : {X.shape[1]} (after feature engineering)')
print(f'  Target   : Early Readmission (<30 days) - 11.2% positive')
print()
print('FEATURE ENGINEERING DONE')
print('  23 drug columns -> binary active flags')
print('  Insulin change flag created')
print('  Total active drugs count feature')
print('  Age groups created')
print('  All categoricals encoded to numbers')
print('  Binary target created')
print()
print('MODEL RESULTS')
print(f'  Logistic Regression ROC-AUC : {lr_auc:.4f}')
print(f'  Random Forest ROC-AUC       : {rf_auc:.4f}')
print(f'  Best Model                  : {"Random Forest" if rf_auc > lr_auc else "Logistic Regression"}')
print()
print('NEXT STEPS')
print('  -> Build a dashboard (Tableau / Power BI / Streamlit)')
print('  -> Write your project report with these charts')
print('  -> Upload to GitHub with a proper README')
print('  -> Add to your resume / portfolio!')

---
## What to Do After This
| Step | Task |
|------|------|
| 17 | Build a dashboard in Tableau or Power BI using your EDA charts |
| 18 | Write project report: Problem, EDA Insights, Model Results, Conclusion |
| 19 | Push everything to GitHub with a proper README.md |
| 20 | Add project link to your resume and LinkedIn! |